# 02 Data Quality and Warehouse

## Project Context

This notebook implements Phase 1 of the Finance Data Engineering Stack —
a portfolio project demonstrating a production-style market data pipeline.

**Disclaimer:** This is a portfolio data engineering project. Nothing here constitutes
investment advice or a production trading system.

Phase 0 produced:
- `dim_assets.csv`, `fact_prices.csv`, `fact_returns.csv` — SQL-ready dimension and fact tables
- `adjusted_close_prices.csv`, `daily_returns.csv` — wide-format analysis tables
- Ingestion log and summary

Phase 1 picks up from those outputs and adds:
1. Data quality validation suite
2. DuckDB embedded warehouse
3. Analytical SQL queries
4. Exported query summaries and figures

## Phase 1 Objectives

1. **Validate** the processed tables for schema correctness, uniqueness, referential integrity,
   missing values, and numeric ranges.
2. **Load** the validated tables into a DuckDB embedded warehouse.
3. **Query** the warehouse with 4 analytical SQL queries and export results as CSV.
4. **Visualise** validation status, return volatility, and latest prices as PNG figures.

In [1]:
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Resolve project root (nbconvert sets CWD to the notebook directory)
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(5):
    if (PROJECT_ROOT / "src").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    PROCESSED_DATA_DIR, WAREHOUSE_DIR, SUMMARIES_DIR, FIGURES_DIR,
)
from src.validation import (
    load_table, run_validation_suite, save_validation_results,
)
from src.warehouse import (
    create_warehouse, create_duckdb_connection,
    run_query, export_query_result,
)

DB_PATH = WAREHOUSE_DIR / "finance_data.duckdb"
RUN_TS  = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

print(f"Project root : {PROJECT_ROOT}")
print(f"Run timestamp: {RUN_TS}")
print(f"DuckDB path  : {DB_PATH}")

Project root : /Users/rovs/Documents/New project 2/projects/finance-data-engineering-stack
Run timestamp: 2026-04-30 12:39:30
DuckDB path  : /Users/rovs/Documents/New project 2/projects/finance-data-engineering-stack/data/warehouse/finance_data.duckdb


## Load Processed Tables

In [2]:
dim_assets   = load_table(PROCESSED_DATA_DIR / "dim_assets.csv")
fact_prices  = load_table(PROCESSED_DATA_DIR / "fact_prices.csv")
fact_returns = load_table(PROCESSED_DATA_DIR / "fact_returns.csv")

print(f"dim_assets   : {dim_assets.shape}")
print(f"fact_prices  : {fact_prices.shape}")
print(f"fact_returns : {fact_returns.shape}")
print()
print(dim_assets.to_string(index=False))

dim_assets   : (9, 4)
fact_prices  : (14301, 4)
fact_returns : (14292, 4)

 asset_id ticker       asset_type           loaded_at
        1   AAPL       Technology 2026-04-30 12:10:40
        2    JNJ      Health Care 2026-04-30 12:10:40
        3    JPM       Financials 2026-04-30 12:10:40
        4     KO Consumer Staples 2026-04-30 12:10:40
        5   MSFT       Technology 2026-04-30 12:10:40
        6   NVDA       Technology 2026-04-30 12:10:40
        7     PG Consumer Staples 2026-04-30 12:10:40
        8    SPY              ETF 2026-04-30 12:10:40
        9    XOM           Energy 2026-04-30 12:10:40


## Data Quality Validation Suite

The validation suite checks:

| Check | Tables |
|---|---|
| Required columns present | all |
| No duplicate composite keys | all |
| Date range coverage | fact_prices, fact_returns |
| Missing values | all |
| Numeric ranges (adj_close > 0; daily_return in [-1, 1]) | fact_prices, fact_returns |
| Referential integrity (ticker FK → dim_assets) | fact_prices, fact_returns |

In [3]:
tables = {
    "dim_assets":   dim_assets,
    "fact_prices":  fact_prices,
    "fact_returns": fact_returns,
}

results = run_validation_suite(tables)
results_df = pd.DataFrame(results)

print(f"Total checks run: {len(results)}")
print()
print(results_df.to_string(index=False))

Total checks run: 15

       table                                check status                                                   details
  dim_assets                     required_columns   PASS                            All 4 required columns present
  dim_assets                    no_duplicate_keys   PASS                               No duplicates on ['ticker']
  dim_assets                       missing_values   PASS                                         No missing values
 fact_prices                     required_columns   PASS                            All 4 required columns present
 fact_prices                    no_duplicate_keys   PASS                       No duplicates on ['date', 'ticker']
 fact_prices                           date_range   PASS Date range: 2020-01-02 to 2026-04-29; 0 unparseable dates
 fact_prices                       missing_values   PASS                                         No missing values
 fact_prices              numeric_range:adj_close   PASS  

## Validation Results Summary

In [4]:
# Save detailed results
results_path  = SUMMARIES_DIR / "data_quality_results.csv"
save_validation_results(results, results_path)

# Build overall summary
pass_count = sum(1 for r in results if r["status"] == "PASS")
fail_count = sum(1 for r in results if r["status"] == "FAIL")
warn_count = sum(1 for r in results if r["status"] == "WARN")

summary = pd.DataFrame([{
    "run_timestamp":   RUN_TS,
    "total_checks":    len(results),
    "pass_count":      pass_count,
    "fail_count":      fail_count,
    "warn_count":      warn_count,
    "overall_status":  "PASS" if fail_count == 0 else "FAIL",
}])

summary_path = SUMMARIES_DIR / "data_quality_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Saved: {results_path.name}  ({len(results)} checks)")
print(f"Saved: {summary_path.name}")
print()
print(summary.T.to_string(header=False))

Saved: data_quality_results.csv  (15 checks)
Saved: data_quality_summary.csv

run_timestamp   2026-04-30 12:39:30
total_checks                     15
pass_count                       15
fail_count                        0
warn_count                        0
overall_status                 PASS


## DuckDB Warehouse Creation

Load `dim_assets`, `fact_prices`, and `fact_returns` from CSV into a local
DuckDB file database at `data/warehouse/finance_data.duckdb`.
DuckDB runs embedded (no server required) and supports full ANSI SQL.

In [5]:
table_paths = {
    "dim_assets":   PROCESSED_DATA_DIR / "dim_assets.csv",
    "fact_prices":  PROCESSED_DATA_DIR / "fact_prices.csv",
    "fact_returns": PROCESSED_DATA_DIR / "fact_returns.csv",
}

create_warehouse(DB_PATH, table_paths)

conn = duckdb.connect(str(DB_PATH))
tables_in_db = conn.execute("SHOW TABLES").fetchdf()
print("Tables in warehouse:")
print(tables_in_db.to_string(index=False))
print()
for tbl in ["dim_assets", "fact_prices", "fact_returns"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {count:,} rows")
conn.close()

Tables in warehouse:
        name
  dim_assets
 fact_prices
fact_returns

  dim_assets: 9 rows
  fact_prices: 14,301 rows
  fact_returns: 14,292 rows


## SQL Table Review

Inspect schema and sample rows for each warehouse table.

In [6]:
conn = duckdb.connect(str(DB_PATH))
for tbl in ["dim_assets", "fact_prices", "fact_returns"]:
    print(f"\n{'='*55}")
    print(f"  {tbl}")
    print(f"{'='*55}")
    schema = conn.execute(f"DESCRIBE {tbl}").fetchdf()
    print(schema[["column_name", "column_type"]].to_string(index=False))
    print("\nSample (3 rows):")
    sample = conn.execute(f"SELECT * FROM {tbl} LIMIT 3").fetchdf()
    print(sample.to_string(index=False))
conn.close()


  dim_assets
column_name column_type
   asset_id      BIGINT
     ticker     VARCHAR
 asset_type     VARCHAR
  loaded_at   TIMESTAMP

Sample (3 rows):
 asset_id ticker  asset_type           loaded_at
        1   AAPL  Technology 2026-04-30 12:10:40
        2    JNJ Health Care 2026-04-30 12:10:40
        3    JPM  Financials 2026-04-30 12:10:40

  fact_prices
column_name column_type
   price_id      BIGINT
       date        DATE
     ticker     VARCHAR
  adj_close      DOUBLE

Sample (3 rows):
 price_id       date ticker  adj_close
        1 2020-01-02   AAPL  72.400505
        2 2020-01-02    JNJ 122.638229
        3 2020-01-02    JPM 118.430344

  fact_returns
 column_name column_type
   return_id      BIGINT
        date        DATE
      ticker     VARCHAR
daily_return      DOUBLE

Sample (3 rows):
 return_id       date ticker  daily_return
         1 2020-01-03   AAPL     -0.009722
         2 2020-01-03    JNJ     -0.011578
         3 2020-01-03    JPM     -0.013197


## Sample Analytical Queries

Run four analytical queries against the DuckDB warehouse:
1. Latest adjusted close price by ticker
2. Return statistics (average daily return, annualised return, annualised volatility)
3. Best and worst single-day returns
4. Joined price and return table (last 5 trading days)

In [7]:
conn = duckdb.connect(str(DB_PATH))

# Query 1: Latest prices
q_latest = """
    SELECT fp.ticker, da.asset_type,
           fp.date AS latest_date,
           ROUND(fp.adj_close, 2) AS latest_price
    FROM fact_prices fp
    JOIN dim_assets da ON fp.ticker = da.ticker
    WHERE fp.date = (SELECT MAX(date) FROM fact_prices)
    ORDER BY fp.ticker
"""
latest_prices = run_query(conn, q_latest)
print("=== Latest Prices ===")
print(latest_prices.to_string(index=False))

# Query 2: Return statistics
q_stats = """
    SELECT fr.ticker, da.asset_type,
           COUNT(*) AS trading_days,
           ROUND(AVG(fr.daily_return) * 100, 4)            AS avg_daily_return_pct,
           ROUND(AVG(fr.daily_return) * 252 * 100, 2)      AS approx_annualised_return_pct,
           ROUND(STDDEV(fr.daily_return) * SQRT(252) * 100, 2) AS annualised_volatility_pct
    FROM fact_returns fr
    JOIN dim_assets da ON fr.ticker = da.ticker
    GROUP BY fr.ticker, da.asset_type
    ORDER BY approx_annualised_return_pct DESC
"""
return_stats = run_query(conn, q_stats)
print("\n=== Return Statistics ===")
print(return_stats.to_string(index=False))

=== Latest Prices ===
ticker       asset_type latest_date  latest_price
  AAPL       Technology  2026-04-29        270.17
   JNJ      Health Care  2026-04-29        227.35
   JPM       Financials  2026-04-29        309.25
    KO Consumer Staples  2026-04-29         78.87
  MSFT       Technology  2026-04-29        424.46
  NVDA       Technology  2026-04-29        209.25
    PG Consumer Staples  2026-04-29        146.46
   SPY              ETF  2026-04-29        711.58
   XOM           Energy  2026-04-29        154.67

=== Return Statistics ===
ticker       asset_type  trading_days  avg_daily_return_pct  approx_annualised_return_pct  annualised_volatility_pct
  NVDA       Technology          1588                0.2783                         70.14                      52.43
  AAPL       Technology          1588                0.1026                         25.84                      31.49
   XOM           Energy          1588                0.0883                         22.24           

In [8]:
# Query 3: Best and worst returns
q_best = """
    SELECT 'best' AS label, date, ticker,
           ROUND(daily_return * 100, 4) AS daily_return_pct
    FROM (
        SELECT date, ticker, daily_return
        FROM fact_returns
        ORDER BY daily_return DESC
        LIMIT 5
    )
"""
q_worst = """
    SELECT 'worst' AS label, date, ticker,
           ROUND(daily_return * 100, 4) AS daily_return_pct
    FROM (
        SELECT date, ticker, daily_return
        FROM fact_returns
        ORDER BY daily_return ASC
        LIMIT 5
    )
"""
best_returns  = run_query(conn, q_best)
worst_returns = run_query(conn, q_worst)
best_worst    = pd.concat([best_returns, worst_returns], ignore_index=True)
print("=== Best and Worst Single-Day Returns ===")
print(best_worst.to_string(index=False))

# Query 4: Joined prices and returns (last 5 trading days)
q_joined = """
    SELECT fp.date, fp.ticker, da.asset_type,
           ROUND(fp.adj_close, 4)          AS adj_close,
           ROUND(fr.daily_return * 100, 4) AS daily_return_pct
    FROM fact_prices fp
    JOIN fact_returns fr ON fp.date = fr.date AND fp.ticker = fr.ticker
    JOIN dim_assets da   ON fp.ticker = da.ticker
    WHERE fp.date >= (SELECT MAX(date) - INTERVAL '5 days' FROM fact_prices)
    ORDER BY fp.date DESC, fp.ticker
"""
joined_sample = run_query(conn, q_joined)
print("\n=== Joined Prices and Returns (last 5 trading days) ===")
print(joined_sample.to_string(index=False))

conn.close()

=== Best and Worst Single-Day Returns ===
label       date ticker  daily_return_pct
 best 2023-05-25   NVDA           24.3696
 best 2025-04-09   NVDA           18.7227
 best 2020-03-13    JPM           18.0125
 best 2020-03-24   NVDA           17.1564
 best 2024-02-22   NVDA           16.4009
worst 2020-03-16   NVDA          -18.4521
worst 2025-01-27   NVDA          -16.9682
worst 2020-03-16    JPM          -14.9649
worst 2020-03-16   MSFT          -14.7390
worst 2020-03-09    JPM          -13.5455

=== Joined Prices and Returns (last 5 trading days) ===
      date ticker       asset_type  adj_close  daily_return_pct
2026-04-29   AAPL       Technology     270.17           -0.1995
2026-04-29    JNJ      Health Care     227.35           -0.1932
2026-04-29    JPM       Financials     309.25           -0.7064
2026-04-29     KO Consumer Staples      78.87            0.6637
2026-04-29   MSFT       Technology     424.46           -1.1159
2026-04-29   NVDA       Technology     209.25          

## Exported Query Outputs

In [9]:
latest_prices.to_csv(SUMMARIES_DIR / "query_latest_prices.csv", index=False)
return_stats.to_csv(SUMMARIES_DIR  / "query_return_stats.csv",  index=False)
best_worst.to_csv(SUMMARIES_DIR    / "query_best_worst_returns.csv", index=False)
joined_sample.to_csv(SUMMARIES_DIR / "query_joined_prices_returns_sample.csv", index=False)

print("Query outputs saved:")
for fname in [
    "query_latest_prices.csv",
    "query_return_stats.csv",
    "query_best_worst_returns.csv",
    "query_joined_prices_returns_sample.csv",
]:
    p = SUMMARIES_DIR / fname
    print(f"  {p.name:<50}  {p.stat().st_size:>8,} bytes")

Query outputs saved:
  query_latest_prices.csv                                  342 bytes
  query_return_stats.csv                                   457 bytes
  query_best_worst_returns.csv                             331 bytes
  query_joined_prices_returns_sample.csv                 1,513 bytes


## Figures

Three PNG figures saved to `reports/figures/`:
1. Validation check status bar chart
2. Annualised volatility by ticker
3. Latest adjusted close price by ticker

In [10]:
STATUS_COLORS = {"PASS": "#22c55e", "FAIL": "#ef4444", "WARN": "#f59e0b"}

# Figure 1: Validation status
status_counts = results_df["status"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    status_counts.index,
    status_counts.values,
    color=[STATUS_COLORS.get(s, "#94a3b8") for s in status_counts.index],
    edgecolor="white",
    linewidth=1.5,
)
ax.bar_label(bars, padding=4, fontsize=13, fontweight="bold")
ax.set_title("Data Quality Validation Status", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Status", fontsize=11)
ax.set_ylabel("Number of Checks", fontsize=11)
ax.set_facecolor("#f9fafb")
ax.spines[["top", "right"]].set_visible(False)
fig.patch.set_facecolor("white")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "data_quality_status.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved: data_quality_status.png")

# Figure 2: Annualised volatility by ticker
vol = return_stats.sort_values("annualised_volatility_pct", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(vol["ticker"], vol["annualised_volatility_pct"],
               color="#6366f1", edgecolor="white", linewidth=1)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=9)
ax.set_title("Annualised Volatility by Ticker (2020-present)",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Annualised Volatility (%)", fontsize=11)
ax.set_facecolor("#f9fafb")
ax.spines[["top", "right"]].set_visible(False)
fig.patch.set_facecolor("white")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "return_volatility_by_ticker.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved: return_volatility_by_ticker.png")

# Figure 3: Latest prices by ticker
pr = latest_prices.sort_values("latest_price", ascending=True)
latest_date_label = str(latest_prices["latest_date"].iloc[0])
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(pr["ticker"], pr["latest_price"],
               color="#0ea5e9", edgecolor="white", linewidth=1)
ax.bar_label(bars, fmt="$%.2f", padding=4, fontsize=9)
ax.set_title(f"Latest Adjusted Close Prices ({latest_date_label})",
             fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Adjusted Close (USD)", fontsize=11)
ax.set_facecolor("#f9fafb")
ax.spines[["top", "right"]].set_visible(False)
fig.patch.set_facecolor("white")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "latest_prices_by_ticker.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved: latest_prices_by_ticker.png")

Saved: data_quality_status.png
Saved: return_volatility_by_ticker.png
Saved: latest_prices_by_ticker.png


## Data Engineering Interpretation

### Validation
All schema, uniqueness, referential integrity, and numeric range checks are expected to pass
for clean yfinance data. WARN status may appear on the missing values check for `loaded_at`
in `dim_assets` if timezone formatting differs across runs — this is benign.

### Warehouse
The DuckDB warehouse uses three tables mirroring a standard star schema:
- `dim_assets` (dimension) — one row per ticker with sector classification
- `fact_prices` (fact) — one row per (date, ticker), adj_close
- `fact_returns` (fact) — one row per (date, ticker), daily_return

This schema supports efficient grouping, joining, and filtering by ticker, sector, and date
without scanning wide matrices.

### Return statistics
NVDA shows the highest annualised return and volatility over 2020-present,
consistent with its role as the primary AI-chip beneficiary.
KO and PG (consumer staples) show the lowest volatility — expected for defensive equities.
SPY sits near the middle, representing the broad market average.

### API
A FastAPI REST layer (`api/app.py`) wraps the DuckDB warehouse with four endpoints
(`/health`, `/assets`, `/latest-prices`, `/return-stats`). The API is read-only
and is intended for local portfolio demonstration, not production deployment.

## Limitations

- **Batch ingestion only:** The pipeline processes a full historical download each run.
  Incremental (append-only) ingestion would require a watermark strategy.
- **No rolling-origin validation:** Train/test splits and model evaluation are not
  part of this phase. The validation layer only checks data schema and ranges.
- **No live data:** yfinance data may lag by one trading day. Real-time data
  would require a paid provider (Bloomberg, Refinitiv, Polygon).
- **No schema migration:** If new tickers are added, `dim_assets` must be regenerated
  and the warehouse rebuilt. A migration strategy is a Phase 2 improvement.
- **Single-file DuckDB:** The `finance_data.duckdb` file is regenerated from CSV each run
  and is excluded from Git. A clone requires re-running both notebooks.
- **Portfolio project disclaimer:** All outputs are for demonstration purposes only.

## Next Steps for Phase 2 Polish

1. Finalise `README.md` with screenshots, architecture diagram reference, and run instructions.
2. Write career-facing reports: `resume_bullets.md`, `interview_talking_points.md`,
   `company_positioning.md`, `linkedin_post.md`.
3. Review and expand `architecture_notes.md` with a Mermaid pipeline diagram.
4. Final commit and GitHub push with a clean, tagged release.
5. Optional: add incremental ingestion logic to `src/ingestion.py`.

In [11]:
# Output verification
check_files = [
    SUMMARIES_DIR / "data_quality_results.csv",
    SUMMARIES_DIR / "data_quality_summary.csv",
    SUMMARIES_DIR / "query_latest_prices.csv",
    SUMMARIES_DIR / "query_return_stats.csv",
    SUMMARIES_DIR / "query_best_worst_returns.csv",
    SUMMARIES_DIR / "query_joined_prices_returns_sample.csv",
    FIGURES_DIR   / "data_quality_status.png",
    FIGURES_DIR   / "return_volatility_by_ticker.png",
    FIGURES_DIR   / "latest_prices_by_ticker.png",
    DB_PATH,
]

print("Phase 1 output verification:")
print("-" * 70)
all_ok = True
for p in check_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if (exists and size > 0) else "MISSING"
    if status != "OK":
        all_ok = False
    rel = p.relative_to(PROJECT_ROOT)
    print(f"  {status:<8}  {size:>10,} bytes  {rel}")
print("-" * 70)
print("All outputs verified." if all_ok else "WARNING: one or more outputs missing.")

Phase 1 output verification:
----------------------------------------------------------------------
  OK             1,115 bytes  outputs/summaries/data_quality_results.csv
  OK               110 bytes  outputs/summaries/data_quality_summary.csv
  OK               342 bytes  outputs/summaries/query_latest_prices.csv
  OK               457 bytes  outputs/summaries/query_return_stats.csv
  OK               331 bytes  outputs/summaries/query_best_worst_returns.csv
  OK             1,513 bytes  outputs/summaries/query_joined_prices_returns_sample.csv
  OK            24,127 bytes  reports/figures/data_quality_status.png
  OK            46,269 bytes  reports/figures/return_volatility_by_ticker.png
  OK            51,777 bytes  reports/figures/latest_prices_by_ticker.png
  OK         1,847,296 bytes  data/warehouse/finance_data.duckdb
----------------------------------------------------------------------
All outputs verified.
